<a href="https://colab.research.google.com/github/mandara-samanalee/FYP_Team_Intellicore/blob/main/notebooks/module3_activity_pattern_analysis/V1/V1_Feature_Extraction_for_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Temporal Feature Extraction for Review Burstiness Detection
This process extracts temporal and sliding-window statistical features from chronological review sequences grouped by product. It helps identify spike patterns, burstiness, and temporal anomalies across product histories.

* **Inputs:**
  * `df_train.csv`: Training dataset containing historical review records with product identifiers and Unix timestamps (`timestamp`).
  * `df_val.csv`: Validation dataset.
  * `df_test.csv`: Test dataset.

* **Outputs:**
  * `df_features_train.csv`: Processed training dataset containing sliding-window statistics per product. Features include:
    * `review_count`: Total reviews within the 30-day window.
    * `mean_iat`: Average Inter-Arrival Time between consecutive reviews.
    * `std_iat`: Standard deviation of Inter-Arrival Times.
    * `burstiness`: Burstiness coefficient indicating regular versus spike behavior.

In [ ]:
# ============================================================
# RESTORE CELL — Run this first every session
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

print("Loading data...")

df_train = pd.read_csv("/content/drive/MyDrive/df_train.csv")
df_val   = pd.read_csv("/content/drive/MyDrive/df_val.csv")
df_test  = pd.read_csv("/content/drive/MyDrive/df_test.csv")

df_train["datetime"] = pd.to_datetime(df_train["datetime"])
df_val["datetime"]   = pd.to_datetime(df_val["datetime"])
df_test["datetime"]  = pd.to_datetime(df_test["datetime"])

print(f"df_train: {len(df_train):,} records")
print(f"df_val:   {len(df_val):,} records")
print(f"df_test:  {len(df_test):,} records")
print("Ready to continue")

Mounted at /content/drive
Loading data...
df_train: 324,446 records
df_val:   69,524 records
df_test:  69,525 records
Ready to continue


In [ ]:
# ============================================================
# CELL — Temporal Feature Extraction
# Sliding window: 30 day window, 7 day step
# Features per product per window:
#   - review_count
#   - mean_inter_arrival_time
#   - std_inter_arrival_time
#   - burstiness_coefficient
# ============================================================

WINDOW_DAYS = 30
STEP_DAYS   = 7

WINDOW_SECS = WINDOW_DAYS * 24 * 3600
STEP_SECS   = STEP_DAYS   * 24 * 3600

print(f"Window size: {WINDOW_DAYS} days")
print(f"Step size:   {STEP_DAYS} days")
print(f"Extracting features from df_train...")
print(f"Total products: {df_train['product_id'].nunique():,}\n")

all_windows = []

products    = df_train["product_id"].unique()
total       = len(products)

for idx, product_id in enumerate(products):

    if idx % 1000 == 0:
        print(f"  Processing product {idx:,} of {total:,}...")

    # Get all reviews for this product sorted by time
    product_df = df_train[
        df_train["product_id"] == product_id
    ].sort_values("timestamp")

    timestamps = product_df["timestamp"].values

    if len(timestamps) == 0:
        continue

    # Define window boundaries
    t_start = timestamps[0]
    t_end   = timestamps[-1]

    # Slide window across product history
    window_start = t_start

    while window_start <= t_end:
        window_end = window_start + WINDOW_SECS

        # Get reviews in this window
        mask = (
            (timestamps >= window_start) &
            (timestamps <  window_end)
        )
        window_timestamps = timestamps[mask]
        window_count      = len(window_timestamps)

        # ── Feature 1 — Review count ──────────────────────
        review_count = window_count

        # ── Features 2, 3, 4 — IAT based ─────────────────
        if window_count >= 2:
            iat            = np.diff(window_timestamps)
            mean_iat       = float(np.mean(iat))
            std_iat        = float(np.std(iat))

            # Burstiness coefficient
            # Range: -1 (perfectly regular) to +1 (very bursty)
            if (mean_iat + std_iat) > 0:
                burstiness = (std_iat - mean_iat) / (
                    std_iat + mean_iat
                )
            else:
                burstiness = 0.0

        else:
            # Option B — fewer than 2 reviews, set to 0
            mean_iat   = 0.0
            std_iat    = 0.0
            burstiness = 0.0

        all_windows.append({
            "product_id":    product_id,
            "window_start":  pd.Timestamp(window_start, unit="s"),
            "window_end":    pd.Timestamp(window_end,   unit="s"),
            "review_count":  review_count,
            "mean_iat":      mean_iat,
            "std_iat":       std_iat,
            "burstiness":    burstiness,
            "is_generated":  0   # all train data is genuine
        })

        # Move window forward by step size
        window_start += STEP_SECS

# ── Create features dataframe ─────────────────────────────
df_features_train = pd.DataFrame(all_windows)

print(f"\nFeature extraction complete")
print(f"Total windows:    {len(df_features_train):,}")
print(f"Unique products:  {df_features_train['product_id'].nunique():,}")
print(f"\nFeature statistics:")
print(df_features_train[[
    "review_count", "mean_iat", "std_iat", "burstiness"
]].describe())
print(f"\nSample:")
print(df_features_train.head())

Window size: 30 days
Step size:   7 days
Extracting features from df_train...
Total products: 13,196

  Processing product 0 of 13,196...
  Processing product 1,000 of 13,196...
  Processing product 2,000 of 13,196...
  Processing product 3,000 of 13,196...
  Processing product 4,000 of 13,196...
  Processing product 5,000 of 13,196...
  Processing product 6,000 of 13,196...
  Processing product 7,000 of 13,196...
  Processing product 8,000 of 13,196...
  Processing product 9,000 of 13,196...
  Processing product 10,000 of 13,196...
  Processing product 11,000 of 13,196...
  Processing product 12,000 of 13,196...
  Processing product 13,000 of 13,196...

Feature extraction complete
Total windows:    893,238
Unique products:  13,196

Feature statistics:
        review_count      mean_iat       std_iat     burstiness
count  893238.000000  8.932380e+05  8.932380e+05  893238.000000
mean        1.456419  1.865705e+05  4.348108e+04      -0.170218
std         4.310309  4.018444e+05  1.314417e

In [ ]:
# ============================================================
# SAVE — Run immediately
# ============================================================

df_features_train.to_csv(
    "/content/drive/MyDrive/df_features_train.csv",
    index=False
)

print(f"Saved successfully")
print(f"Total windows: {len(df_features_train):,}")
print(f"Unique products: {df_features_train['product_id'].nunique():,}")

Saved successfully
Total windows: 893,238
Unique products: 13,196
